# U2.L1 — Solvers and adjoints in practice

**Session U2.L1** (lab, joint). Companion to the U2.L1 lecture notes, which carry
the reasoning behind every block; this notebook is the part you execute.

U2.T1 proved the adjoint theorem and wrote a table of two gradient strategies whose
cells were arguments. Today every cell becomes a measurement, on a small Neural ODE
that classifies the two-moons points. By the end you own an **instrument panel**:

1. an **NFE counter** (three lines);
2. **NFE-against-quality axes**, on two metrics that disagree;
3. **NFE drift** across training;
4. a **peak-memory protocol** for one forward-and-backward pass;
5. a **gradient-fidelity table**: two differentiation routes against one witness;
6. **honest timing**: compile time reported apart from steady state.

Nothing here is about generative models. The next lab (U2.L2) points these
instruments at one.

### How to read this notebook

Cells marked **TODO** are yours. Each TODO is followed by a **Checkpoint** cell that
either passes or tells you what is wrong. Everything else is given code: run it,
read it, move on. The solutions notebook, `lab-u2l1-solutions.ipynb`, which the
instructor releases, carries the filled-in cells.

**Everything runs on CPU**, on one thread, in a few minutes. The memory measurement
of Block 3 uses a GPU if one is present; without one it falls back to a coarse
reading, and says so. Block 4 needs JAX and Diffrax, both pinned in the course
environment.

**Library documentation:** torchdiffeq, <https://github.com/rtqichen/torchdiffeq>;
Diffrax, <https://github.com/patrick-kidger/diffrax> (docs:
<https://docs.kidger.site/diffrax/>).

## Block 0 — Orientation, and the entry quiz

### Given code

| Module | Built in | What this lab uses |
|---|---|---|
| `common/toy2d.py` | U1.L1 (frozen) | `two_moons(n, seed)`, a `(n, 2)` float32 tensor |
| `common/odesolve.py` | U0.L5 (frozen) | `odeint(f, x0, t0, t1, method="rk4", n_steps=N)`, whose result carries `.nfe` |
| `common/train.py` | U0.L1, U0.L2 | `set_seed(seed)` only |

You do not need to have attended the bootcamp sessions that built two of these; their
contracts are all this lab uses.

**Two moons as a classification task.** `two_moons` returns points without labels.
Its frozen contract puts the first `n // 2` rows on the upper moon and the rest on
the lower moon, so the label is the row order. The frozen file does not change.

**The training loop is written out**, not the given `Trainer`: the NFE counter must
be read between the forward solve and the backward pass of every step, and the
`Trainer` has no hook there.

### Where the numbers go

MLflow, deployed locally: a SQLite file next to this notebook
(`sqlite:///mlflow.db`). Experiment `U2`, one run per block, tagged
`session=U2.L1` and `block=<n>`, with the configuration as parameters and the git
commit and dirty flag as tags. The curves are logged **after** each loop, in one
batch: the loop measures the wall-clock of every step (a median of 53 ms on the
reference machine), and a SQLite write inside the loop would become part of what it
measures. To look at the runs afterwards:

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [ ]:
import sys
from pathlib import Path

# The repository root, so `import common.toy2d` works from labs/.
ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(ROOT))

import hashlib
import math
import statistics
import subprocess
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdiffeq import odeint, odeint_adjoint

from common import odesolve
from common.toy2d import EXTENT, two_moons
from common.train import set_seed

torch.set_num_threads(1)   # tiny operations: one thread is fastest, and it reproduces the notes
set_seed(0)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
import mlflow
from mlflow.entities import Metric

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("U2")


def _git(*args):
    out = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
    return out.stdout.strip()


def log_block(block, params, curves=None, metrics=None):
    """One MLflow run per block, logged AFTER the work, in one batch per 1000 values."""
    with mlflow.start_run(run_name=f"u2l1-block{block}") as run:
        mlflow.set_tags({"session": "U2.L1", "block": str(block),
                         "git_commit": _git("rev-parse", "HEAD"),
                         "git_dirty": str(bool(_git("status", "--porcelain")))})
        mlflow.log_params({k: str(v) for k, v in params.items()})
        batch = [Metric(name, float(v), 0, step)
                 for name, values in (curves or {}).items() for step, v in enumerate(values)]
        client = mlflow.tracking.MlflowClient()
        for i in range(0, len(batch), 1000):
            client.log_batch(run.info.run_id, metrics=batch[i:i + 1000])
        if metrics:
            mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
    print(f"MLflow: experiment U2, run u2l1-block{block}, {len(batch)} curve values")

### The entry quiz

This lab assumes, without teaching it again, what the bootcamp packet's solver notes
say about adaptive solvers (U0.T5, *Adaptive stepping* and *Stability*). Answer on
paper, then check against the answers at the **end of this notebook**. With four or
more right, go on. With fewer, read those two sections before Block 2; Block 1 does
not need them.

1. What does NFE count, and why does the course report it for every solve?
2. An adaptive solver takes `rtol` and `atol`. Where does each one dominate?
3. Why does an adaptive solver reject a step, and what does a rejected step cost?
4. What is `dopri5`?
5. For a fixed-step method, what decides the cost of a solve? For an adaptive one?
6. What does it mean, loosely, for an equation to be *stiff*?

## Block 1 — A Neural ODE classifier on two moons

An input point $x_0 \in \mathbb{R}^2$ is the initial state of
$\dot x(t) = f_\theta(t, x(t))$ on $t \in [0, 1]$; the flow carries it to
$x(1) = \psi_1(x_0)$; a linear head reads the class: logits $= W\,\psi_1(x_0) + b$.
The loss is the cross-entropy averaged over the training set,

$$
\ell(\theta) = \mathbb{E}_{(x_0, y) \sim \mathcal{D}_{\mathrm{train}}}
\Big[ -\log \operatorname{softmax}\big(W\,\psi_1(x_0) + b\big)_{y} \Big].
$$

Two choices are deliberate. **The state stays in the plane**, so a picture of the
trajectories is a picture of the state, not a projection. **The activation is smooth**:
with $\tanh$ the field is continuously differentiable in $(t, x, \theta)$ and globally
Lipschitz in $x$, which is both tiers of U2.T1's regularity contract; Tier 2 is what the
adjoint theorem assumes.

In [ ]:
x_train = two_moons(512, 0)
y_train = (torch.arange(512) >= 256).long()        # 0 = upper moon, 1 = lower moon
x_test = two_moons(2048, 1)
y_test = (torch.arange(2048) >= 1024).long()

# the labels read from the row order are the moons: label 0 sits higher
m0, m1 = float(x_train[y_train == 0, 1].mean()), float(x_train[y_train == 1, 1].mean())
print(f"mean second coordinate: label 0 {m0:+.2f}, label 1 {m1:+.2f}")
assert m0 > m1 + 1.0

### The field (given)

The right-hand side takes the time first, `f(t, x)`, as every solver of the course
does. The solver passes `t` as a scalar tensor; the network sees it as a column
beside `x`. The last layer starts at zero (recipe card item 3), so the flow starts at
the identity.

In [ ]:
class ODEField(nn.Module):
    """The right-hand side f_theta(t, x): R^2 -> R^2 of the Neural ODE.

    The solver calls it as f(t, x), time first (U0.T5), with t a scalar
    tensor. The network sees t as a (B,) column beside x: the course
    convention for time in a network signature. tanh is smooth with
    |tanh'| <= 1, so the field is C^1 in (t, x, theta) and globally Lipschitz
    in x: both tiers of U2.T1's regularity contract.
    """

    def __init__(self, hidden: int = 32) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2),
        )
        # Recipe card item 3: zero-init the last layer, so f = 0 at the start
        # and the flow map starts at the identity.
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, t: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        tb = t.to(x.dtype).expand(x.shape[0])                  # (B,)
        return self.net(torch.cat([x, tb[:, None]], dim=1))

### TODO 1.1 — The NFE counter

The number of function evaluations is the cost currency of this course. You build the
gauge yourself, once: a wrapper module that holds the field and counts every call. Three
lines. Because it is an `nn.Module` holding the field, the field's parameters are its
parameters, and `odeint_adjoint` finds them in Block 3 without an `adjoint_params` argument.

In [ ]:
class NFECounter(nn.Module):
    """Wraps a field and counts its calls: the NFE, the course's cost currency."""

    def __init__(self, f: nn.Module) -> None:
        super().__init__()
        self.f = f
        # TODO 1.1 (line 1): start the count at zero, in an attribute named nfe.
        raise NotImplementedError

    def forward(self, t: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        # TODO 1.1 (lines 2-3): count this call, then return the field's value.
        raise NotImplementedError

### The classifier (given)

The head starts at zero as well, so the first prediction is uniform and the first loss
is $\log 2$. The notes explain why that was measured, not assumed: with PyTorch's default
head, some seeds stall at every learning rate tried.

In [ ]:
class NODEClassifier(nn.Module):
    """x(0) is the input point, x(1) = psi_1(x(0)), and the logits are W x(1) + b."""

    def __init__(self, hidden: int = 32, zero_head: bool = True) -> None:
        super().__init__()
        self.field = NFECounter(ODEField(hidden))
        self.head = nn.Linear(2, 2)
        if zero_head:
            # Recipe card item 3 again: logits = 0 at the start, a uniform first
            # prediction (loss log 2). With PyTorch's default head, some seeds
            # stall at every learning rate tried (measured: stage `init` of
            # scripts/python/fig_u2_l1_instruments.py).
            nn.init.zeros_(self.head.weight)
            nn.init.zeros_(self.head.bias)

    def flow(self, x0: torch.Tensor, ts: torch.Tensor | None = None,
             adjoint: bool = False, **solver) -> torch.Tensor:
        """The states x(t) at the times ts (default: t = 0 and t = 1)."""
        if ts is None:
            ts = torch.tensor([0.0, 1.0], dtype=x0.dtype, device=x0.device)
        solve = odeint_adjoint if adjoint else odeint
        return solve(self.field, x0, ts, **solver)             # (len(ts), B, 2)

    def forward(self, x0: torch.Tensor, adjoint: bool = False, **solver) -> torch.Tensor:
        return self.head(self.flow(x0, adjoint=adjoint, **solver)[-1])

In [ ]:
# Checkpoint 1.1 - the counter against arithmetic the solver cannot fake
model = NODEClassifier(hidden=32)
with torch.no_grad():
    for method, n, expected in (("euler", 8, 8), ("rk4", 8, 32), ("rk4", 16, 64)):
        model.field.nfe = 0
        model.flow(x_train, method=method, options={"step_size": 1 / n})
        print(f"{method:5s} N={n:3d}: NFE {model.field.nfe}")
        assert model.field.nfe == expected, (method, n, model.field.nfe)
print("Checkpoint 1.1 passed.")

### TODO 1.2 — Train with the solver's defaults, and read the gauges

The solver is `dopri5` at torchdiffeq's defaults (`rtol=1e-7`, `atol=1e-9`). The
optimizer is the U0.T2 recipe card: AdamW, weight decay 0.01, linear warmup over 3% of
the steps then cosine decay, stepped per step, full batch.

**One deviation, in writing, with its reason:** the base learning rate is `3e-3`, not
the card's `3e-4`. Measured on the reference machine, same model and seed: trained for
1500 steps at `3e-4` (77 s), the training loss ends at 0.0275 (and is still
0.287 at step 300 of that run); 300 steps at `3e-3` (15 s) end at
0.0032, 8.5 times lower. Five times the steps do not close the gap.

Your loop body reads the counter **twice**: after the forward solve (`nfe_fwd`), and after
`loss.backward()` (the calls the backward pass made, `nfe_bwd`).

In [ ]:
CONFIG = {"hidden": 32, "steps": 300, "lr": 3e-3, "weight_decay": 0.01,
          "warmup_frac": 0.03, "method": "dopri5", "rtol": 1e-7, "atol": 1e-9}


def warmup_cosine(step, total, warmup):
    """Multiplier on the base learning rate: the U0.T2 schedule, per step."""
    if step < warmup:
        return step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * p))

In [ ]:
def train(seed, steps=CONFIG["steps"], lr=CONFIG["lr"], stop_after=None):
    """Full-batch training; returns the model and the per-step history."""
    set_seed(seed)
    model = NODEClassifier(hidden=CONFIG["hidden"])     # built before any data draw
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG["weight_decay"])
    warm = max(1, int(CONFIG["warmup_frac"] * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: warmup_cosine(s, steps, warm))
    hist = {"loss": [], "acc": [], "nfe_fwd": [], "nfe_bwd": [], "sec": []}
    for step in range(stop_after or steps):
        t0 = time.perf_counter()
        # TODO 1.2: reset the counter; forward with method="dopri5" (defaults);
        # read nfe_fwd; cross-entropy; zero_grad; backward; opt.step(); sched.step();
        # read nfe_bwd; append loss, accuracy, nfe_fwd, nfe_bwd and the seconds.
        raise NotImplementedError
    return model, hist

In [ ]:
model, hist = train(seed=0)
model.eval()
with torch.no_grad():
    model.field.nfe = 0
    acc_test = float((model(x_test, method="dopri5").argmax(1) == y_test).float().mean())
    nfe_test = model.field.nfe
print(f"{sum(hist['sec']):.1f} s, {1e3 * statistics.median(hist['sec']):.0f} ms per step")
print(f"final training loss {hist['loss'][-1]:.4f}; test accuracy {100 * acc_test:.1f}%")
print(f"forward NFE: step 1 {hist['nfe_fwd'][0]}, last step {hist['nfe_fwd'][-1]}; "
      f"test solve {nfe_test}")
print(f"backward NFE, largest over all steps: {max(hist['nfe_bwd'])}")

log_block(1, CONFIG, curves={k: hist[k] for k in ("loss", "acc", "nfe_fwd", "nfe_bwd")},
          metrics={"test_acc": acc_test, "test_nfe": nfe_test})

In [ ]:
# Checkpoint 1.2
assert acc_test >= 0.99, f"test accuracy {acc_test:.3f}: check the loop body against the notes"
assert max(hist["nfe_bwd"]) == 0, "direct backprop should never call the field in backward"
print("Checkpoint 1.2 passed. (Reference run: test NFE 116; yours may differ by a few.)")

### The rescue checkpoint

Blocks 2–4 need a trained model. If your Block 1 run is missing or failed, set
`USE_REFERENCE = True`: the cell loads the reference weights, the ones behind every
number in the notes, and checks the file's SHA-256 first.

In [ ]:
USE_REFERENCE = False
CKPT = ROOT / "common" / "weights" / "u2l1-node-classifier.pt"


def load_reference():
    sha = hashlib.sha256(CKPT.read_bytes()).hexdigest()
    assert sha == "c9527e8b5e712cc32277c3afc97c3d10dd6fef88d47471cda9e637ae92947c9b", "the checkpoint on disk is not the one the notes describe"
    m = NODEClassifier(hidden=CONFIG["hidden"])
    m.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))
    return m.eval()


trained = load_reference() if USE_REFERENCE else model
print("Blocks 2-4 use:", "the reference checkpoint" if USE_REFERENCE else "your Block 1 model")

### Checkpoint 1 — The trajectories, in the plane

Under U2.T1's Tier 1 (Picard–Lindelöf), two distinct trajectories never occupy the same
state at the same time. The cell reads that theorem off the model, with a stated floor: over
pairs of test points and 41 times, the ratio $|x^{(i)}(t) - x^{(j)}(t)| / |x^{(i)}(0) - x^{(j)}(0)|$
must stay above 0.1 (the reference run measures 0.41).

Read panel (b) with one caution. The field depends on $t$, so two drawn *paths* may cross:
a crossing of paths is two trajectories at the same place at **different** times.

In [ ]:
with torch.no_grad():
    t_grid = torch.linspace(0.0, 1.0, 41)
    traj = trained.flow(x_test[::4], ts=t_grid, method="dopri5")   # 512 test points
    d0 = torch.cdist(traj[0], traj[0])
    off = ~torch.eye(d0.shape[0], dtype=torch.bool)
    ratio = min(float((torch.cdist(traj[k], traj[k])[off] / d0[off]).min()) for k in range(41))
print(f"smallest equal-time distance ratio over {int(off.sum()) // 2} pairs: {ratio:.3f}")
assert ratio > 0.1, f"equal-time ratio {ratio:.3f}: below the stated floor"

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
g = torch.linspace(EXTENT[0], EXTENT[1], 121)
gx, gy = torch.meshgrid(g, g, indexing="xy")
with torch.no_grad():
    pred = trained(torch.stack([gx.ravel(), gy.ravel()], 1), method="dopri5").argmax(1)
axes[0].contourf(gx, gy, pred.reshape(gx.shape), levels=[-0.5, 0.5, 1.5],
                 colors=["#dbe4ef", "#f7e3cf"])
colors = ("#003366", "#d97a1f")
for c in (0, 1):
    pts = x_test[::5][y_test[::5] == c]
    axes[0].scatter(pts[:, 0], pts[:, 1], s=6, color=colors[c])
axes[0].set_aspect("equal")
axes[0].set_title("(a) inputs x(0) and decision regions")
paths = traj.numpy()
for j in range(0, paths.shape[1], 13):
    c = colors[int(y_test[::4][j])]
    axes[1].plot(paths[:, j, 0], paths[:, j, 1], color=c, lw=1)
    axes[1].scatter(paths[-1, j, 0], paths[-1, j, 1], color=c, s=12)
axes[1].set_aspect("equal")
axes[1].set_title("(b) states x(t), t in [0, 1]")
plt.show()

### Optional warmup — your RK4 against the library's

Skip this first when time is short. Integrate the trained field with the bootcamp's own
solver (`common/odesolve.py`, classical RK4) on a fixed grid, and compare the endpoints
with a tight library reference. A caution the numbers will show: torchdiffeq's `rk4` is
the **3/8-rule** fourth-order method, not the classical tableau. Both are order 4; on the
same grid they are two different computations.

In [ ]:
# The float64 copy of the trained model, and a tight reference solve of the test set.
# Given, and used by the optional warmup below AND by the sweep of Block 2.
f64 = NODEClassifier(hidden=32)
f64.load_state_dict(trained.state_dict())
f64 = f64.double().eval()
x64 = x_test.double()
with torch.no_grad():
    x_ref = f64.flow(x64, method="dopri5", rtol=1e-12, atol=1e-14)[-1]
print("reference solve ready: dopri5, rtol=1e-12, atol=1e-14, float64")

In [ ]:
with torch.no_grad():
    for n in (8, 32, 64):
        ours = odesolve.odeint(f64.field.f, x64, 0.0, 1.0, method="rk4", n_steps=n)
        lib = f64.flow(x64, method="rk4", options={"step_size": 1 / n})[-1]
        print(f"N={n:3d}  odesolve {float((ours - x_ref).abs().max()):.2e}  "
              f"torchdiffeq rk4 {float((lib - x_ref).abs().max()):.2e}  "
              f"between them {float((ours - lib).abs().max()):.2e}  NFE {ours.nfe}")
    err64 = float((odesolve.odeint(f64.field.f, x64, 0.0, 1.0, method="rk4", n_steps=64)
                   - x_ref).abs().max())
# threshold calibrated on the reference run (ten times its error, rounded up)
assert err64 < 0.0001, err64
print("Optional checkpoint passed.")

## Block 2 — Solver and tolerance sweep: NFE as the cost currency

The sweep (TODOs 2.1 and 2.2) is evaluation only: the model is fixed. TODO 2.3 re-trains a
second seed for 100 steps. The sweep runs in **float64**,
because it is a measurement (train in float32, promote the check). In float32 the tightest
settings would measure round-off, not the solver.

> **A hypothesis to test, not a law.** The task metric may saturate before the trajectory
> error does. If it does, tighter tolerances spend evaluations without changing the
> classifier's answer.

### TODO 2.1 — The sweep

For `euler` and `rk4` with $N \in \{4, 8, 16, 32\}$ fixed steps, and for `dopri5` with
`rtol` in $\{10^{-1}, \dots, 10^{-7}\}$ and `atol = rtol * 1e-2`: record the NFE, the
test accuracy, and the RMS error of $x(1)$ against `x_ref` (the tight reference solve of
Block 1).

In [ ]:
@torch.no_grad()
def evaluate(m, x, y, **solver):
    """One forward solve: NFE, accuracy, and the terminal states."""
    m.field.nfe = 0
    z1 = m.flow(x, **solver)[-1]
    return {"nfe": m.field.nfe, "acc": float((m.head(z1).argmax(1) == y).float().mean()), "z1": z1}

In [ ]:
rows = []
# TODO 2.1: for each setting, call evaluate(f64, x64, y_test, **solver) and append a dict
# {"family": ..., "setting": ..., "nfe": ..., "acc": ..., "err": RMS of |z1 - x_ref|}.
# Fixed step: solver = dict(method=m, options={"step_size": 1 / n}).
# Adaptive:   solver = dict(method="dopri5", rtol=r, atol=r * 1e-2).
raise NotImplementedError

In [ ]:
# Checkpoint 2.1 - fifteen rows, and the fixed-step counts are arithmetic
assert len(rows) == 15
for r in rows:
    if r["family"] == "euler":
        assert r["nfe"] == r["setting"]
    if r["family"] == "rk4":
        assert r["nfe"] == 4 * r["setting"]
for r in rows:
    print(f"{r['family']:6s} {r['setting']:>7g}  NFE {r['nfe']:4d}  "
          f"accuracy {100 * r['acc']:6.2f}%  error {r['err']:.2e}")
print("Checkpoint 2.1 passed.")

### TODO 2.2 — Two axes that disagree

Draw (a) test accuracy against NFE, and (b) the terminal-state error against NFE on a log
scale, one line per family; then (c) the `dopri5` NFE against `rtol`. Then answer below:
did the hypothesis survive?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
# TODO 2.2: panel (a) accuracy vs NFE, panel (b) error vs NFE (log-log),
# panel (c) dopri5 NFE vs rtol (log x-axis, inverted). One line per family.
raise NotImplementedError

**Your reading (TODO 2.2, in two sentences):** did the task metric saturate before the
terminal error did? What does that say about which gauge to trust here?

### One step size for the whole batch (given)

torchdiffeq's step controller reduces the error estimate to one number: a root mean square
over **every** element of the batched state. The cell compares the NFE of the whole test
batch with the NFE of 64 single points, spread over both moons (every 32nd row: the first
1024 rows are all upper moon), solved alone.

In [ ]:
batch_nfe = evaluate(f64, x64, y_test, method="dopri5")["nfe"]
singles = [evaluate(f64, x64[i:i + 1], y_test[i:i + 1], method="dopri5")["nfe"] for i in range(0, 2048, 32)]
print(f"whole batch: NFE {batch_nfe}")
print(f"64 single points: NFE {min(singles)} to {max(singles)}, median {statistics.median(singles):.0f}")

### TODO 2.3 — NFE across training

A solver setting fixes a tolerance, not a cost; the cost follows the field, and the field
changes at every step. Re-train from a fresh initialization with **seed 1**, for the first
100 steps of the same 300-step schedule (`stop_after=100`), and plot its forward NFE per
step beside the first 100 steps of your Block 1 run. Then **describe** what happened.
Nothing in advance says what you will see.

In [ ]:
# TODO 2.3: _, hist1 = train(seed=1, stop_after=100); plot hist["nfe_fwd"][:100] and
# hist1["nfe_fwd"] against the step, and log the new run with log_block(2, ...).
raise NotImplementedError

**What happened (TODO 2.3, in words):** describe the curve. Why might the first three steps
cost what they cost? (Hint: what is the field at those steps, and what is the learning rate
at step 1?)

## Block 3 — Memory and gradient fidelity

The empirical companion to U2.T1's table of the two gradient strategies. Both routes use
fixed-step `rk4`, so the forward work is identical and only the backward pass differs:
`odeint` (direct backprop through the solver) against `odeint_adjoint` (the continuous
adjoint of Box U2.T1-3).

### The protocol

- **GPU (canonical).** `torch.cuda.max_memory_allocated`, reset and synchronized around
  each run, minus what was resident before.
- **`tracemalloc` is not a tensor meter.** It sees Python's allocator; PyTorch allocates
  tensor storage in native code that it does not see.
- **CPU fallback.** Without a GPU, the cell samples the process's resident set size every
  millisecond. That reading is **coarse** and not a contract: the allocator reuses memory
  the process already holds.

In [ ]:
def one_backward(m, x, y, n_steps, adjoint):
    m.zero_grad(set_to_none=True)
    m.field.nfe = 0
    logits = m(x, adjoint=adjoint, method="rk4", options={"step_size": 1 / n_steps})
    nfe_fwd = m.field.nfe
    F.cross_entropy(logits, y).backward()
    return nfe_fwd, m.field.nfe - nfe_fwd


def peak_gpu(m, x, y, n_steps, adjoint):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    nfe = one_backward(m, x, y, n_steps, adjoint)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() - base, nfe


def peak_rss(m, x, y, n_steps, adjoint):
    """COARSE: process RSS sampled every millisecond (psutil). Not a contract."""
    import psutil
    proc = psutil.Process()
    base = proc.memory_info().rss
    peak, stop = [base], threading.Event()

    def sample():
        while not stop.is_set():
            peak[0] = max(peak[0], proc.memory_info().rss)
            time.sleep(0.001)

    th = threading.Thread(target=sample, daemon=True)
    th.start()
    nfe = one_backward(m, x, y, n_steps, adjoint)
    stop.set()
    th.join()
    return peak[0] - base, nfe


GPU = torch.cuda.is_available()
meter = peak_gpu if GPU else peak_rss
dev = "cuda" if GPU else "cpu"
mem_model = NODEClassifier(hidden=32)
mem_model.load_state_dict(trained.state_dict())
mem_model = mem_model.to(dev)
xm, ym = x_train.to(dev), y_train.to(dev)
one_backward(mem_model, xm, ym, 8, False)       # warm the kernels once
one_backward(mem_model, xm, ym, 8, True)
print("meter:", "GPU allocator (canonical)" if GPU else "process RSS (COARSE)")

### TODO 3.1 — Peak memory against the number of steps

For $N \in \{8, 16, 32, 64, 128, 256\}$ and both routes, measure the peak with `meter`,
record the forward and backward NFE, and plot the peak (MiB) against the forward NFE on
log-log axes. Compare with U2.T1's schematic.

In [ ]:
mem_rows = []
# TODO 3.1: for n in (8, 16, 32, 64, 128, 256), for adjoint in (False, True):
#   peak, (nfe_fwd, nfe_bwd) = meter(mem_model, xm, ym, n, adjoint)
#   append a dict; then plot peak / 2**20 against nfe_fwd, one line per route.
raise NotImplementedError

In [ ]:
# Checkpoint 3.1
direct = {r["n"]: r for r in mem_rows if r["route"] == "direct"}
adj = {r["n"]: r for r in mem_rows if r["route"] == "adjoint"}
assert all(r["nfe_bwd"] == 0 for r in direct.values()), "direct backprop never calls the field backward"
assert all(r["nfe_bwd"] == r["nfe_fwd"] for r in adj.values()), "adjoint on a fixed grid: one backward call per forward call"
if GPU:
    assert direct[256]["mib"] / direct[8]["mib"] > 10, "direct backprop should grow with the steps"
    assert adj[256]["mib"] / adj[8]["mib"] < 1.5, "the adjoint should not grow with the steps"
    print("Checkpoint 3.1 passed (GPU reading).")
else:
    print("CPU fallback: no memory assertion; the NFE assertions passed. Read the plot as coarse.")
log_block(3, {"meter": "gpu" if GPU else "rss", "batch": 512, "method": "rk4"},
          metrics={f"mib_{r['route']}_{r['n']}": r["mib"] for r in mem_rows})

### TODO 3.2 — The gradient-fidelity table

The course ritual (U0.L1): float64, one fixed unit direction $v$, and a central difference
as the witness that owes nothing to automatic differentiation. On a fixed batch $B$ of 256
test points, $\ell_B$ is the loss of the **exact** flow and $\ell_N$ the loss **as the
computer evaluates it**, through $N$ steps of `rk4`. $v$ is drawn over the field's
parameters only (the head's gradient passes through no solve).

$$
d_{\mathrm{FD}} = \frac{\ell_N(\theta + \varepsilon v) - \ell_N(\theta - \varepsilon v)}{2\varepsilon},
\qquad d_{\mathrm{AD}} = v^\top \nabla_{\theta_f} \ell_N \ \text{(direct backprop)},
$$

$d_{\mathrm{ADJ}} = v^\top \hat g_N$, where $\hat g_N$ is the adjoint's estimate of
$\nabla_{\theta_f} \ell_B$ (its backward solve on the same $N$-step grid), and
$d_{\mathrm{cont}} = v^\top \nabla_{\theta_f} \ell_B$, with $\theta_f$ the field's parameters.

Write `directional` (autodiff along $v$) and `central_difference` (restore the parameters
afterwards!), fill the table for $N \in \{8, 32\}$, and report absolute and relative gaps.

In [ ]:
fid = NODEClassifier(hidden=32)
fid.load_state_dict(trained.state_dict())
fid = fid.double()
xb, yb = x_test[::8].double(), y_test[::8]                  # 256 fixed test points
gen = torch.Generator().manual_seed(2026)
v = [torch.randn(p.shape, generator=gen, dtype=torch.float64) for p in fid.field.parameters()]
vnorm = torch.sqrt(sum((t ** 2).sum() for t in v))
v = [t / vnorm for t in v]

In [ ]:
def directional(m, x, y, v, adjoint, **solver):
    """v . grad of the loss, over the field's parameters, by autodiff."""
    # TODO 3.2a: zero the grads, backward the cross-entropy of m(x, adjoint=..., **solver),
    # and return sum((p.grad * t).sum()) over zip(m.field.parameters(), v).
    raise NotImplementedError


@torch.no_grad()
def central_difference(m, x, y, v, eps=1e-6, **solver):
    """The witness: two forward solves, parameters moved along +-eps v, then restored."""
    # TODO 3.2b
    raise NotImplementedError

In [ ]:
# the exact flow's gradient, two ways that must agree
d_cont = directional(fid, xb, yb, v, False, method="dopri5", rtol=1e-12, atol=1e-14)
d_cont2 = directional(fid, xb, yb, v, False, method="rk4", options={"step_size": 1 / 1024})
print(f"d_cont = {d_cont:.12e}  (two routes differ by {abs(d_cont - d_cont2):.1e})")

fid_rows = []
for n in (8, 32):
    solver = {"method": "rk4", "options": {"step_size": 1 / n}}
    d_fd = central_difference(fid, xb, yb, v, **solver)
    d_ad = directional(fid, xb, yb, v, False, **solver)
    d_adj = directional(fid, xb, yb, v, True, **solver)
    fid_rows.append((n, d_fd, d_ad, d_adj))
    print(f"N={n:3d}  |AD-FD| {abs(d_ad - d_fd):.1e} (rel {abs(d_ad - d_fd) / abs(d_fd):.0e})  "
          f"|ADJ-FD| {abs(d_adj - d_fd):.1e} (rel {abs(d_adj - d_fd) / abs(d_fd):.0e})  "
          f"|AD-cont| {abs(d_ad - d_cont):.1e}  |ADJ-cont| {abs(d_adj - d_cont):.1e}")

In [ ]:
# Checkpoint 3.2 - the ritual on direct backprop, at both step sizes
for n, d_fd, d_ad, d_adj in fid_rows:
    tol = 1e-8 + 1e-6 * max(abs(d_ad), abs(d_fd))
    assert abs(d_ad - d_fd) <= tol, (n, d_ad, d_fd)
    assert abs(d_adj - d_fd) > abs(d_ad - d_fd), "the adjoint answers a different question"
print("Checkpoint 3.2 passed: direct autodiff matches the witness; the adjoint does not, by design.")
log_block(3, {"part": "fidelity", "eps": 1e-6, "batch": 256},
          metrics={f"rel_adj_fd_{n}": abs(d_adj - d_fd) / abs(d_fd) for n, d_fd, _, d_adj in fid_rows})

**Your reading (TODO 3.2):** which route is exact for which function? Which one is closer to
the exact flow's gradient here?

**Margin note — the backward solve can drift.** The adjoint reconstructs $x(t)$ by solving
the forward equation backward. Where the forward flow contracts, the backward flow
amplifies error (U2.T1). Mitigations: tighter tolerances on the backward solve (paid in
NFE); direct backprop when the problem is small enough to store; checkpointing, which
stores a few states and reconstructs only between them. Block 4 shows a library that makes
the last one its default.

## Block 4 — The Diffrax mirror

**If time runs out, this block goes home whole;** it is self-contained.

**The honest-timing rule (U0.L5), restated so that it survives the trip home:** any claim
about the speed of a JAX function reports **compile time separately** from steady-state
time, and never amortises the compile into the speedup. Cross-framework wall-clock
numbers are descriptive only: the hardware, the compiler and the solver all differ.

The same classifier in JAX and Diffrax, loaded with the weights of Block 1. JAX runs on CPU
here, in float64. The block needs the setup cells of Blocks 0 and 1 (or the rescue
checkpoint) and the first cell of TODO 3.2, which fixes the batch and the direction $v$.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import diffrax
import jax.numpy as jnp

net = trained.field.f.net
P = {k: jnp.asarray(t.detach().double().numpy()) for k, t in
     zip(["W1", "b1", "W2", "b2", "W3", "b3"],
         [net[0].weight, net[0].bias, net[2].weight, net[2].bias, net[4].weight, net[4].bias])}
HEAD = {"W": jnp.asarray(trained.head.weight.detach().double().numpy()),
        "b": jnp.asarray(trained.head.bias.detach().double().numpy())}
X0, Y = jnp.asarray(xb.numpy()), jnp.asarray(yb.numpy())


def field(t, x, p):
    tb = jnp.full((x.shape[0], 1), t)
    h = jnp.tanh(jnp.concatenate([x, tb], axis=1) @ p["W1"].T + p["b1"])
    h = jnp.tanh(h @ p["W2"].T + p["b2"])
    return h @ p["W3"].T + p["b3"]


TERM = diffrax.ODETerm(field)


def terminal(p, x, n_steps, adjoint):
    sol = diffrax.diffeqsolve(TERM, diffrax.Dopri5(), t0=0.0, t1=1.0, dt0=1.0 / n_steps, y0=x,
                              args=p, stepsize_controller=diffrax.ConstantStepSize(),
                              saveat=diffrax.SaveAt(t1=True), adjoint=adjoint,
                              max_steps=4 * n_steps)
    return sol.ys[-1]


def loss(p, n_steps, adjoint):
    logits = terminal(p, X0, n_steps, adjoint) @ HEAD["W"].T + HEAD["b"]
    return -jnp.mean(jax.nn.log_softmax(logits)[jnp.arange(Y.shape[0]), Y])


# check 1: the port computes what torch computes (tight adaptive solves on both sides)
sol = diffrax.diffeqsolve(TERM, diffrax.Dopri5(), t0=0.0, t1=1.0, dt0=None, y0=X0, args=P,
                          stepsize_controller=diffrax.PIDController(rtol=1e-12, atol=1e-14),
                          saveat=diffrax.SaveAt(t1=True), max_steps=100_000)
with torch.no_grad():
    z_torch = fid.flow(xb, method="dopri5", rtol=1e-12, atol=1e-14)[-1].numpy()
gap = float(np.abs(np.asarray(sol.ys[-1]) - z_torch).max())
print(f"JAX vs torch terminal states: {gap:.1e}")
assert gap < 1e-9

# check 2: Diffrax reports steps, not calls; count the calls with a host callback
calls = [0]


def counted(t, x, p):
    jax.debug.callback(lambda: calls.__setitem__(0, calls[0] + 1))
    return field(t, x, p)


sol_c = diffrax.diffeqsolve(diffrax.ODETerm(counted), diffrax.Dopri5(), t0=0.0, t1=1.0,
                            dt0=1.0 / 8, y0=X0, args=P, stepsize_controller=diffrax.ConstantStepSize(),
                            saveat=diffrax.SaveAt(t1=True), max_steps=32)
jax.block_until_ready(sol_c.ys)
print(f"Dopri5, {int(sol_c.stats['num_steps'])} steps: {calls[0]} calls")

### TODO 4.1 — Honest timing

Diffrax exposes the differentiation route as one argument, `adjoint=`. Its default is
`diffrax.RecursiveCheckpointAdjoint()` (discretize, then optimize: backpropagation through
the solver, with checkpointing to keep memory low). Another is
`diffrax.BacksolveAdjoint()` (optimize, then discretize: the continuous adjoint of U2.T1),
which Diffrax's documentation marks "not recommended", because its gradients are
approximate. (Diffrax has further routes; these two are the ones this lab compares.)

For each route and $N \in \{8, 32\}$: build `jax.jit(jax.value_and_grad(...))`, **lower and
compile it explicitly** and time that step, then time the median of 20 calls with
`jax.block_until_ready`.

In [ ]:
ADJOINTS = {"RecursiveCheckpointAdjoint": diffrax.RecursiveCheckpointAdjoint(),
            "BacksolveAdjoint": diffrax.BacksolveAdjoint()}
timing = []
# TODO 4.1: for name, adj in ADJOINTS.items(), for n in (8, 32):
#   vg = jax.jit(jax.value_and_grad(lambda p: loss(p, n, adj)))
#   time vg.lower(P) AND .compile() together (the whole up-front price); warm once;
#   then the median of 20 timed calls.
raise NotImplementedError

### TODO 4.2 — Two philosophies, one metric

Apply the directional metric of Block 3 to both routes: the same direction $v$ (moved into
JAX), the same batch, a central difference of the **same** discrete program (fixed-step
`Dopri5` with $N$ steps). Report the relative gap of each route to its witness.

In [ ]:
names = ["W1", "b1", "W2", "b2", "W3", "b3"]
V = {k: jnp.asarray(t.numpy()) for k, t in zip(names, v)}
# TODO 4.2: for n in (8, 32): d_fd from two jitted loss evaluations at P +- 1e-6 V;
# for each route, d = sum of jnp.vdot(grads[(name, n)][k], V[k]); print |d - d_fd| / |d_fd|.
raise NotImplementedError

In [ ]:
# Checkpoint 4 - the checkpointed route agrees with its own program; backsolve does not
for n in (8, 32):
    rc = next(r for r in jax_rows if r[0] == "RecursiveCheckpointAdjoint" and r[1] == n)
    bs = next(r for r in jax_rows if r[0] == "BacksolveAdjoint" and r[1] == n)
    assert bs[2] > rc[2], (bs, rc)
print("Checkpoint 4 passed.")
log_block(4, {"framework": f"jax {jax.__version__}, diffrax {diffrax.__version__}", "device": "cpu"},
          metrics={f"{name}_{n}_steady_ms": ms for name, n, _, ms in timing}
          | {f"{name}_{n}_compile_s": c for name, n, c, _ in timing})

**Closing: two libraries, two emphases.** Diffrax defaults to checkpointed backpropagation;
torchdiffeq still exposes the classical continuous adjoint as its prominent memory-saving
route, and there the route is the function you call, `odeint` or `odeint_adjoint`. Neither is
wrong; they weigh one trade differently. Knowing which route you run is part of reading any
paper that says "we trained with the adjoint". For the numerics behind Diffrax's defaults,
and for adjoint numerics beyond this course: Kidger, *On Neural Differential Equations*
(PhD thesis, 2022), chapter 5 (optional).

## Block 5 — Debrief: the instrument panel

| Instrument | Where you built it |
|---|---|
| NFE counter | TODO 1.1 (training part) |
| NFE-against-quality axes | TODO 2.2 (sweep part) |
| NFE drift in training | TODO 2.3 (sweep part) |
| Peak-memory protocol | TODO 3.1 (profiling part) |
| Gradient-fidelity table | TODO 3.2 (profiling part) |
| Honest timing | TODO 4.1 (Diffrax part) |

Every gauge on this panel gets pointed at a generative model in the FFJORD lab — and
there, the needles move. Watch them.

## Answers to the entry quiz

1. **NFE** is the number of evaluations of the field $f(t, x)$ during a solve. The field
   is the expensive part (a network call), so two solvers are comparable only at a stated
   NFE.
2. The error scale per component is `atol + rtol * max(|x_n|, |x_{n+1}|)`. **`rtol`**
   dominates where the solution is large (significant digits); **`atol`** dominates near
   zero (it stops the solver from demanding infinite precision there).
3. The solver estimates each step's local error with an embedded pair. If the normalized
   error exceeds 1, the step is **rejected**, thrown away and retried smaller. A rejected
   step costs its evaluations and makes no progress.
4. **`dopri5`** is Dormand–Prince: an embedded Runge–Kutta pair of orders 5 and 4, seven
   stages, the last reused as the first of the next step (six new evaluations per step).
   It is torchdiffeq's default method; Diffrax offers it as `diffrax.Dopri5` and has no
   default solver.
5. **Fixed step:** the new evaluations per step times $N$, plus a start-up call for a
   method that reuses its last stage; known in advance. **Adaptive:** it depends
   on the field, the initial state and the tolerances, and is known only after the solve.
6. **Stiff**, loosely: stability, not accuracy, forces an explicit solver to take small
   steps, because a fast component that has already decayed still sets the step size.